# === ЭТАП 3 ===

In [1]:
import logging
import sys

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

log_file = logging.FileHandler('recommendations.log', mode='w')
log_file.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))

console_out = logging.StreamHandler(sys.stdout)
console_out.setFormatter(logging.Formatter('%(message)s'))

logging.basicConfig(handlers=(log_file, console_out), level=logging.INFO)

# Загрузка данных

Если необходимо, то загружаем items.parquet, events.parquet.

In [2]:
items = pd.read_parquet('../data/items.parquet')
events = pd.read_parquet('../data/events.parquet')

In [3]:
events.shape

(13023329, 8)

In [4]:
import sklearn.preprocessing

# перекодируем идентификаторы пользователей: 
# из имеющихся в последовательность 0, 1, 2, ...
user_encoder = sklearn.preprocessing.LabelEncoder()
user_encoder.fit(events['user_id'])
events['user_id_enc'] = user_encoder.transform(events['user_id'])

# перекодируем идентификаторы объектов: 
# из имеющихся в последовательность 0, 1, 2, ...
item_encoder = sklearn.preprocessing.LabelEncoder()
item_encoder.fit(items['item_id'])
items['item_id_enc'] = item_encoder.transform(items['item_id'])
events['item_id_enc'] = item_encoder.transform(events['item_id'])

# Разбиение данных

Разбиваем данные на тренировочную, тестовую выборки.

In [5]:
train_split_date = pd.to_datetime('2022-12-16').to_datetime64()

train_split_date_idx = events['started_at'] < train_split_date

events_train = events[train_split_date_idx]
events_test = events[~train_split_date_idx]

events_train.shape, events_test.shape

((9110491, 10), (3912838, 10))

In [6]:

# количество пользователей в train и test
users_train = events_train['user_id'].drop_duplicates()
users_test = events_test['user_id'].drop_duplicates()
# количество пользователей, которые есть и в train, и в test
common_users = set(users_train) & set(users_test)

len(users_train), len(users_test), len(common_users)

(1113710, 702781, 456576)

# Похожие

Рассчитаем похожие, они позже пригодятся для онлайн-рекомендаций.

In [7]:
genres = pd.DataFrame(items['genre'].apply(eval).explode('genre').to_list()) \
    .rename(columns={'track_id': 'item_id'})
    
genres['item_id_enc'] = item_encoder.transform(genres['item_id'])

genre_encoder = sklearn.preprocessing.LabelEncoder()
genre_encoder.fit(genres['id'])
genres['id_enc'] = genre_encoder.transform(genres['id'])


In [8]:
import scipy
import sklearn.preprocessing

def get_item2genre_matrix(items: pd.DataFrame):

    # encoder = sklearn.preprocessing.LabelEncoder()
    # encoder.fit(items['item_id'])
    # items['item_id_enc'] = encoder.transform(items['item_id'])
    
    data = items['votes'].to_list()
    row = items['item_id_enc'].to_list()
    col = items['id_enc'].to_list()


    print(data)
    # print(row)

    # list to build CSR matrix
    # genres_csr_data = []
    # genres_csr_row_idx = []
    # genres_csr_col_idx = []
    
    # for _, data in genres.groupby('item_id'):
    
    
    #     # genre_idx = genre_names_to_id[genre_name]
    #     # genres_csr_data.append(int(votes))
    #     genres_csr_row_idx.append(data.item_id)
    #     # genres_csr_col_idx.append(genre_idx)
    
    
    genres_csr = scipy.sparse.csr_matrix((data, (row, col)))
    # нормализуем, чтобы сумма оценок принадлежности к жанру была равна 1
    genres_csr = sklearn.preprocessing.normalize(genres_csr, norm='l1', axis=1)    

    return genres_csr 

all_items_genres_csr = get_item2genre_matrix(genres)

[3684587248434, 704716535880, 433806232924, 3684587248434, 704716535880, 899359023625, 13165499529, 4846390650052, 1668981114810, 9241782868620, 1243604885204, 348019389960, 9241782868620, 52117028594, 9241782868620, 3684587248434, 704716535880, 9241782868620, 1243604885204, 52117028594, 433806232924, 841149379467, 348019389960, 243837047512, 3684587248434, 704716535880, 348019389960, 3684587248434, 704716535880, 243837047512, 26400049224, 2298427117008, 3684587248434, 704716535880, 433806232924, 9241782868620, 1243604885204, 348019389960, 9241782868620, 1243604885204, 4846390650052, 1668981114810, 9241782868620, 59087654028, 3684587248434, 704716535880, 2298427117008, 841149379467, 44365757679, 348019389960, 348019389960, 58818960912, 243837047512, 26400049224, 3684587248434, 704716535880, 29407583924, 841149379467, 9241782868620, 1243604885204, 9241782868620, 1243604885204, 9241782868620, 1243604885204, 9241782868620, 1243604885204, 841149379467, 9241782868620, 1243604885204, 3684587

In [9]:
all_items_genres_csr

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 65937 stored elements and shape (39847, 107)>

In [10]:
all_items_genres_csr.toarray().sum()

39846.99999999999

In [11]:
genres.sort_values('item_id_enc')

,item_id,id,name,votes,rating,rank,item_id_enc,id_enc
58503,376,11,pop,9241782868620,10.00,1,0,10
38815,549,68,electronics,2298427117008,8.98,2,1,57
38814,549,11,pop,9241782868620,10.00,1,1,10
23624,553,74,rnb,52117028594,6.71,3,2,61
23623,553,68,electronics,2298427117008,8.98,2,2,57
...,...,...,...,...,...,...,...,...
4365,100904719,20,ruspop,1243604885204,8.57,2,39844,19
46504,100923044,20,ruspop,1243604885204,8.57,2,39845,19
46503,100923044,11,pop,9241782868620,10.00,1,39845,10
57664,101016326,20,ruspop,1243604885204,8.57,2,39846,19


In [12]:
# Аналогичным образом получим матрицу с весами по жанрам для какого-нибудь пользователя

user_id = 592791
user_events = events.query('user_id == @user_id')[['item_id', 'rating']]
user_genres = genres[genres['item_id'].isin(user_events['item_id'])]

user_genres_csr = all_items_genres_csr[user_genres['item_id_enc'].unique()]



In [13]:
user_ratings = user_events['rating'].to_numpy() / 10
user_ratings = np.expand_dims(user_ratings, axis=1)

user_items_genres_weighted = user_genres_csr.multiply(user_ratings)

user_genres_scores = np.asarray(user_items_genres_weighted.mean(axis=0)) 

In [14]:
user_genres

,item_id,id,name,votes,rating,rank,item_id_enc,id_enc
0,53404,102,allrock,3684587248434,9.32,1,701,71
1,53404,14,rock,704716535880,8.20,2,701,13
2,53404,13,alternative,433806232924,7.90,3,701,12
69,148345,102,allrock,3684587248434,9.32,1,1562,71
70,148345,14,rock,704716535880,8.20,2,1562,13
243,96079,102,allrock,3684587248434,9.32,1,1073,71
244,96079,14,rock,704716535880,8.20,2,1073,13
322,38633712,102,allrock,3684587248434,9.32,1,21283,71
323,38633712,2,rusrock,482526121403,7.96,2,21283,1
538,53412,13,alternative,433806232924,7.90,1,709,12


In [15]:
from sklearn.metrics.pairwise import cosine_similarity

# вычисляем сходство между вектором пользователя и векторами по книгам
similarity_scores = cosine_similarity(all_items_genres_csr, user_genres_scores)

# преобразуем в одномерный массив
similarity_scores = similarity_scores.flatten()

# получаем индексы top-k (по убыванию значений), по сути, индексы треков (encoded)
k = 5
top_k_indices = np.argsort(similarity_scores)[-5:]
top_k_indices


array([ 3485, 33417,  7114,  2253,  2479])

In [16]:
similarity_scores

array([0.        , 0.03525807, 0.03525754, ..., 0.        , 0.        ,
       0.        ])

# i2i

# Построение признаков

Построим три признака, можно больше, для ранжирующей модели.

# Ранжирование рекомендаций

Построим ранжирующую модель, чтобы сделать рекомендации более точными. Отранжируем рекомендации.

# Оценка качества

Проверим оценку качества трёх типов рекомендаций: 

- топ популярных,
- персональных, полученных при помощи ALS,
- итоговых
  
по четырем метрикам: recall, precision, coverage, novelty.

# === Выводы, метрики ===

Основные выводы при работе над расчётом рекомендаций, рассчитанные метрики.